In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
import pandas as pd
import os
import numpy as np
import json
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

REPO = "microsoft/rad-dino"

PROJECT_DIR = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge"
IMG_DIR = os.path.join(PROJECT_DIR, "3000imagenes")
CSV_PATH = os.path.join(PROJECT_DIR, "df_subset_2994_valido.csv")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "resultados-rad-dino-test")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print(f"CSV cargado: {len(df)} filas")

C:\Users\trodr\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CSV cargado: 2994 filas


In [2]:
# Para probar con pocas imágenes, descomentar:
#df = df.head(100)
print(f"Subset: {len(df)} filas")

# Validar que existan las imágenes (usar 'full_path')
existe = df["full_path"].apply(lambda p: os.path.exists(os.path.join(IMG_DIR, p.replace("/", os.sep))))
df = df[existe].reset_index(drop=True)
print(f"Imágenes válidas: {len(df)}")

Subset: 2994 filas
Imágenes válidas: 2994


In [3]:
def cargar_rad_dino():
    processor = AutoImageProcessor.from_pretrained(REPO)
    model = AutoModel.from_pretrained(REPO).to(DEVICE)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, processor

rad_dino, processor = cargar_rad_dino()
print("✓ RAD-DINO cargado y congelado")

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 4153.07it/s]


✓ RAD-DINO cargado y congelado


In [4]:
class DatasetExtraccion(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Usar 'full_path' directamente
        path = os.path.join(self.img_dir, row['full_path'].replace("/", os.sep))
        
        try:
            img = Image.open(path).convert("RGB")
        except:
            img = Image.new('RGB', (512, 512), color='white')
        
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0)

In [5]:
@torch.inference_mode()
def extraer_embeddings(df, img_dir, model, processor, batch_size=32):
    print(f"Extrayendo embeddings para {len(df)} imágenes...")
    dataset = DatasetExtraccion(df, img_dir, processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    embeddings = []
    for batch_idx, batch in enumerate(loader):
        batch = batch.to(DEVICE)
        outputs = model(pixel_values=batch)
        cls_emb = outputs.pooler_output  # (batch, 768)
        embeddings.append(cls_emb.cpu())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  {min((batch_idx + 1) * batch_size, len(df))} / {len(df)}")

    return torch.cat(embeddings, dim=0)

In [6]:
print("\n" + "="*60)
print("EXTRAYENDO EMBEDDINGS CLS (768-dim)")
print("="*60)

embeddings = extraer_embeddings(df, IMG_DIR, rad_dino, processor, batch_size=32)
print(f"✓ Embeddings: shape {embeddings.shape}")

# Guardar
embeddings_path = os.path.join(OUTPUT_DIR, "embeddings.pt")
torch.save(embeddings, embeddings_path)
print(f"✓ Guardado: {embeddings_path}")


EXTRAYENDO EMBEDDINGS CLS (768-dim)
Extrayendo embeddings para 2994 imágenes...
  320 / 2994
  640 / 2994
  960 / 2994
  1280 / 2994
  1600 / 2994
  1920 / 2994
  2240 / 2994
  2560 / 2994
  2880 / 2994
✓ Embeddings: shape torch.Size([2994, 768])
✓ Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\resultados-rad-dino-test\embeddings.pt


In [7]:
class ClassifierHead(nn.Module):
    def __init__(self, in_dim=768, n_labels=13, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_labels)
        )

    def forward(self, x):
        return self.net(x)

print("✓ ClassifierHead definido")

✓ ClassifierHead definido


In [8]:
## CAMBIOS
print("\n" + "="*60)
print("CREANDO SPLITS (TRAIN/TEST) — 80/20")
print("="*60)

from sklearn.model_selection import GroupShuffleSplit

def hacer_splits(df, test_size=0.2, random_state=42):
    """
    Devuelve POSICIONES (0..N-1) alineadas con el tensor de embeddings.
    NO se resetea el índice: los embeddings se extrajeron en el mismo
    orden de las filas de df, así que la posición i del tensor == fila i del df.

    Partición en dos vías (80/20), igual que CNN Simple y DenseNet-121:
    el 20% de test cumple doble función (selección de la mejor época
    vía early stopping y reporte de métricas finales).
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_pos, test_pos = next(gss.split(df, groups=df["subject_id"]))
    return train_pos, test_pos

train_idx, test_idx = hacer_splits(df)

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

# --- Verificaciones de seguridad ---
# 1) No hay solapamiento de posiciones
assert len(set(train_idx) & set(test_idx)) == 0, "LEAK: train/test solapan"

# 2) No hay pacientes compartidos entre splits
subj = df["subject_id"].values
s_train = set(subj[train_idx]); s_test = set(subj[test_idx])
print("Pacientes solapados train/test (debe ser 0):", len(s_train & s_test))


CREANDO SPLITS (TRAIN/TEST) — 80/20
Train: 2396, Test: 598
Pacientes solapados train/test (debe ser 0): 0


In [9]:
class DatasetEmbeddings(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

print("✓ DatasetEmbeddings definido")

✓ DatasetEmbeddings definido


In [10]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix
import mlflow

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax",
]

EPOCHS = 30
LEARNING_RATE = 1e-3
POS_WEIGHT_SCALE = 0.75


# ---- MISMA función evaluate() del CNN, adaptada a embeddings ----
def evaluate(head, loader, criterion, device, label_cols, umbral=0.5):
    head.eval()
    loss_total = 0.0
    todas_probs, todas_lbls = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = head(xb)
            loss_total += criterion(logits, torch.nan_to_num(yb, nan=0.0).clamp(0, 1)).mean().item()

            probs = torch.sigmoid(logits)
            todas_probs.append(probs.cpu())
            todas_lbls.append(yb.cpu())

    y_prob = torch.cat(todas_probs).numpy()
    y_prob = np.nan_to_num(y_prob, nan=0.0)
    y_true = torch.cat(todas_lbls).numpy()

    # NaN -> 0, igual que el CNN
    y_true = np.nan_to_num(y_true, nan=0.0)
    y_true = np.clip(y_true, 0, 1)
    y_pred = (y_prob >= umbral).astype(int)

    # AUC por etiqueta
    aucs = {}
    for i, nombre in enumerate(label_cols):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        aucs[nombre] = roc_auc_score(y_true[:, i], y_prob[:, i])

    # Precision, recall, F1 macro
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    prec_arr, rec_arr, f1_arr, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    # TP, FP, FN, TN por etiqueta
    confusion_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        tn, fp, fn, tp = confusion_matrix(
            y_true[:, i], y_pred[:, i], labels=[0, 1]
        ).ravel()
        confusion_por_etiqueta[nombre] = {
            "TP": int(tp), "FP": int(fp),
            "FN": int(fn), "TN": int(tn)
        }

    metricas_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        metricas_por_etiqueta[nombre] = {
            "auc":       aucs.get(nombre, float("nan")),
            "precision": float(prec_arr[i]),
            "recall":    float(rec_arr[i]),
            "f1":        float(f1_arr[i]),
            **confusion_por_etiqueta[nombre],
        }

    metricas = {
        "loss": loss_total / len(loader),
        "auc": np.mean(list(aucs.values())) if aucs else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    return metricas, aucs, metricas_por_etiqueta


def entrenar_y_evaluar(embeddings, df, train_idx, test_idx, label_cols,
                        n_epochs=30, lr=1e-3, condicion="raddino_sin_limpieza"):
    # Partición 80/20 (igual que CNN Simple y DenseNet-121): el conjunto de
    # test cumple doble función, selecciona la mejor época (early stopping)
    # y reporta las métricas finales. No hay conjunto de validación aparte.

    y = df[label_cols].values
    train_labels = torch.tensor(np.nan_to_num(y[train_idx], nan=0.0), dtype=torch.float32)
    n_pos = (train_labels == 1).sum(dim=0).clamp(min=1)
    n_neg = (train_labels == 0).sum(dim=0).clamp(min=1)
    pos_weight = (n_neg / n_pos).to(DEVICE) * POS_WEIGHT_SCALE

    train_ds = DatasetEmbeddings(embeddings[train_idx], y[train_idx])
    test_ds  = DatasetEmbeddings(embeddings[test_idx],  y[test_idx])

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)

    head = ClassifierHead(in_dim=768, n_labels=len(label_cols)).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    mlflow.set_experiment("torax-raddino")
    with mlflow.start_run(run_name=f"rad-dino-{condicion}"):
        mlflow.log_params({
            "modelo": "RAD-DINO_frozen",
            "condicion": condicion,
            "n_etiquetas": len(label_cols),
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "lr": lr,
            "num_epochs": n_epochs,
        })

        mejor_auc = 0.0
        mejor_epoch = 0
        paciencia = 5  # ← NUEVO
        contador_paciencia = 0  # ← NUEVO
       
        for epoch in range(1, n_epochs + 1):
            # --- train ---
            head.train()
            train_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                yb_clean = torch.nan_to_num(yb, nan=0.0).clamp(0, 1)
                optimizer.zero_grad()
                loss = criterion(head(xb), yb_clean)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)

            # --- eval en test (selecciona la mejor época, igual que CNN/DenseNet) ---
            metricas, aucs_detalle, metricas_por_etiqueta = evaluate(
                head, test_loader, criterion, DEVICE, label_cols
            )

            print(f"Época {epoch:2d}/{n_epochs} | "
                  f"train_loss: {train_loss:.4f} | "
                  f"test_loss: {metricas['loss']:.4f} | "
                  f"test_AUC: {metricas['auc']:.4f}")

            mlflow.log_metrics({
                "train_loss": train_loss,
                "test_loss": metricas["loss"],
                "test_auc": metricas["auc"],
                "test_precision": metricas["precision"],
                "test_recall": metricas["recall"],
                "test_f1": metricas["f1"],
            }, step=epoch)

            if metricas["auc"] > mejor_auc:
                mejor_auc = metricas["auc"]
                mejor_epoch = epoch
                contador_paciencia = 0  # ← NUEVO: reset cuando hay mejora
                torch.save(head.state_dict(), "mejor_head_raddino.pth")
                print(f"   ↑ nuevo mejor AUC: {mejor_auc:.4f} (época {mejor_epoch})")
            else:  # ← NUEVO
                contador_paciencia += 1
                if contador_paciencia >= paciencia:
                    print(f"   → Early stopping en época {epoch} (sin mejora en {paciencia} épocas)")
                    break
        # --- evaluar en TEST con el mejor modelo ---
        head.load_state_dict(torch.load("mejor_head_raddino.pth"))
        metricas_test, aucs_test, metricas_por_etiqueta_test = evaluate(
            head, test_loader, criterion, DEVICE, label_cols
        )

        # log por etiqueta
        for label, auc_val in aucs_test.items():
            if not np.isnan(auc_val):
                mlflow.log_metric(f"test_auc_{label}", auc_val)

        mlflow.log_metrics({
            "test_auc_final": metricas_test["auc"],
            "test_loss_final": metricas_test["loss"],
            "mejor_test_auc": mejor_auc,
            "mejor_epoch": mejor_epoch,
        })

        # CSV artifact
        df_por_etiqueta = pd.DataFrame(metricas_por_etiqueta_test).T
        df_por_etiqueta.to_csv("metricas_por_etiqueta_raddino.csv")
        mlflow.log_artifact("metricas_por_etiqueta_raddino.csv")

    return metricas_test, aucs_test, metricas_por_etiqueta_test

In [11]:
print("\n" + "="*80)
print("EJECUTANDO PIPELINE")
print("="*80)
##Cambio
# Entrenar y evaluar en TEST
metricas_test, aucs_test, metricas_por_etiqueta_final = entrenar_y_evaluar(
    embeddings, df, train_idx, test_idx, LABEL_COLS,
    n_epochs=EPOCHS, lr=LEARNING_RATE, condicion="raddino_sin_limpieza"
)

# --- Print final (mismo formato del CNN) ---
print(f"\n{'='*60}")
print(f"  RESULTADOS FINALES — TEST")
print(f"{'='*60}")
print(f"  Test AUC:    {metricas_test['auc']:.4f}")
print(f"{'='*60}")
print(f"\n{'Etiqueta':<30} {'AUC':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'TN':>4}")
print(f"{'-'*74}")
for label, vals in metricas_por_etiqueta_final.items():
    auc_str = f"{vals['auc']:.4f}" if not np.isnan(vals['auc']) else "  nan"
    print(f"  {label:<28} {auc_str:>6} {vals['precision']:>6.4f} {vals['recall']:>6.4f} {vals['f1']:>6.4f} {vals['TP']:>4} {vals['FP']:>4} {vals['FN']:>4} {vals['TN']:>4}")
print(f"{'='*60}\n")


EJECUTANDO PIPELINE


2026/08/30 13:32:20 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Época  1/30 | train_loss: 0.9759 | test_loss: 0.9903 | test_AUC: 0.7307
   ↑ nuevo mejor AUC: 0.7307 (época 1)
Época  2/30 | train_loss: 0.7331 | test_loss: 1.0251 | test_AUC: 0.7500
   ↑ nuevo mejor AUC: 0.7500 (época 2)
Época  3/30 | train_loss: 0.5980 | test_loss: 1.1177 | test_AUC: 0.7495
Época  4/30 | train_loss: 0.5013 | test_loss: 1.3138 | test_AUC: 0.7431
Época  5/30 | train_loss: 0.4282 | test_loss: 1.4026 | test_AUC: 0.7414
Época  6/30 | train_loss: 0.3767 | test_loss: 1.5601 | test_AUC: 0.7391
Época  7/30 | train_loss: 0.3409 | test_loss: 1.7661 | test_AUC: 0.7340
   → Early stopping en época 7 (sin mejora en 5 épocas)

  RESULTADOS FINALES — TEST
  Test AUC:    0.7496

Etiqueta                          AUC   Prec Recall     F1   TP   FP   FN   TN
--------------------------------------------------------------------------
  Atelectasis                  0.7857 0.4626 0.5862 0.5171   68   79   48  403
  Cardiomegaly                 0.7758 0.4591 0.5368 0.4949   73   86   63  37